<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/miniprojects/MP11B/MP11B_sensor_fusion.ipynb)

*Open this notebook in Google Colab with a GPU runtime (Runtime → Change runtime type → T4 GPU). Its first code cell fetches the library files from the public course repository.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# MP11B · Fusion of Camera, Depth Camera and LiDAR

*Mini project of the course **Deep Learning for Engineering**, Part 2, in one notebook of five parts:*

**A** The Problem · **B** The Data and the Physics · **C** The Reference Model and the Ground Truth · **D** The Network · **E** The Report

*Parts A to C are given and solved: read them, run them, and use what they build. Parts D and E are yours: the TODO cells.*

In [ ]:
# files-cell (mini project) ----------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
EXERCISE = (['course_core.py', 'pinn_core.py', 'problem.py'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex11-vision-navigation/")
PROJECT = (['mp11b.py', 'MP11B_truth.npz'], "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/miniprojects/MP11B/")
ON_COLAB = "google.colab" in sys.modules
for files, url in (EXERCISE, PROJECT):
    for f in files:
        if ON_COLAB or not os.path.exists(f):
            if not ON_COLAB and os.path.exists(os.path.join("..", "..", "exercises", "Ex11-vision-navigation", f)):
                import shutil; shutil.copy(os.path.join("..", "..", "exercises", "Ex11-vision-navigation", f), f)
            else:
                urllib.request.urlretrieve(url + f, f)
            print("fetched", f)
for _m in ['course_core', 'pinn_core', 'problem', 'mp11b', 'r2twin']:
    sys.modules.pop(_m, None)
print("files ready")

In [ ]:
# The twin of the ROSMASTER R2 is MP11A's r2twin.py, shared by both projects: fetched beside this notebook
import os, shutil, sys, urllib.request
if ON_COLAB or not os.path.exists("r2twin.py"):
    if not ON_COLAB and os.path.exists(os.path.join("..", "MP11A", "r2twin.py")):
        shutil.copy(os.path.join("..", "MP11A", "r2twin.py"), "r2twin.py")
    else:
        urllib.request.urlretrieve(PROJECT[1].replace("/MP11B/", "/MP11A/") + "r2twin.py", "r2twin.py")
    print("fetched r2twin.py")
sys.modules.pop("r2twin", None)

In [ ]:
# --- setup ---------------------------------------------------------------
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import course_core as cc
import problem as hs
import mp11b as mp
torch.set_default_dtype(torch.float32)                   # no second derivative here: single precision, as in Ex11
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # the GPU, when the runtime has one
cc.OUTPUT_DIR = "MP11B_outputs"                          # where the report and its figures are written
print("device:", DEVICE, "| torch", torch.__version__)

def middle(fn, reps=5):
    """The middle of five wall-clock timings of fn(), after one warm-up run;
    on a GPU, synchronised so the time is the computation's, not the launch's."""
    sync = torch.cuda.synchronize if torch.cuda.is_available() else (lambda: None)
    fn(); sync(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); sync(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

pb = hs                                                  # Ex11's problem.py, by its name in Ex11
import r2twin as rt                                      # the ROSMASTER R2 twin (MP11A's r2twin.py, fetched above)

---

# Part A · The Problem

## A1 · What is computed, and why an engineer wants it

Ex11 showed that one camera cannot see everything: its network missed grey boxes the colour of the floor, and its dark-pixel rule stopped for shadows. Real robots therefore carry **several sensors that fail differently** - a camera, a depth camera, a LiDAR - and combine what they report. Combining them is called **sensor fusion**, and doing it well is harder than it looks: Ex11's mini project proposal found that three simple rules joined by OR, each sensor allowed to call the path blocked, still collided on 12 of 20 hard courses.

This mini project fuses the three sensors of the **Yahboom ROSMASTER R2** - its colour camera, its depth camera and its LiDAR - into **one network** that answers, ten times a second, *is the path blocked* and *where does the obstacle stand*, in **difficult conditions**: a dim room with sun on the floor, more grey boxes, more low blocks that pass under the LiDAR's scan, more glossy black obstacles that return little light. The car drives one lap of a closed track at 1.5 m/s and must not touch anything.

Everything runs in Colab on MP11A's **digital twin** of the R2 (`r2twin.py`): the camera frame, the depth image and the LiDAR scan are all computed from where the car stands, so the true answer to every question is known. No real car and no ROS are needed (the lecturer, 27 October 2026).

## A2 · The car and its three sensors

| sensor | what it returns here | what it measures | vendor's numbers |
|---|---|---|---|
| **colour camera** (in the Astra Pro Plus) | a 64 × 112 frame, red-green-blue, 0 - 255 | light reflected by surfaces: colour and brightness | 66.1 × 40.2 degrees, 1920 × 1080, 30 frames a second |
| **depth camera** (Astra Pro Plus) | a 48 × 64 image of distances along the camera's axis, metres, 0 where there is no reading | the distance, by a projected infrared pattern and triangulation (structured light) | 0.6 - 8 m, 58.4 × 45.5 degrees, 640 × 480, about 3 mm at 1 m |
| **LiDAR** (SLAM A1, Slamtec RPLIDAR A1) | 360 ranges, one a degree round the car, metres, 0 where nothing returns | the time a laser pulse takes to come back, in one horizontal plane | 0.15 - 12 m, 5.5 turns a second |

The R2 has no ultrasonic sensor; Ex11's proposal had one, and it is not modelled here. The cell prints the twin's table - what the vendor states and what is **ASSUMED** - and shows the three sensors at one place on a test track.

In [ ]:
# The ROSMASTER R2 twin, and its three sensors at one place
for what, value, source in rt.spec():
    print(f"{what:>19}  {value:<72} {source}")
tr = mp.track(1000, 3)                                   # a test track of this project at noise level 3
so = tr.obstacle_s[0][0] - 1.1                           # 1.1 m before its first obstacle
x, y = tr.point(so, 0.05)
rt.show_sensors(tr, x, y, tr.heading(so), speed=1.5)
plt.show()
ax = rt.view3d(tr, x, y, tr.heading(so), title="the ROSMASTER R2 twin and its LiDAR's scan plane")
plt.show()

**What you should see.** The twin's table (nineteen rows, VENDOR or ASSUMED). Then the three sensors on test track 1000 at noise level 3: the dim, grainy colour frame; the depth image, distances coloured from yellow (near) to purple (far) with white holes where no reading came back (sun patches, glossy surfaces, beyond 8 m, nearer than 0.6 m); and the LiDAR scan from above, red dots where a beam returned, the dashed box the zone in which an obstacle blocks the path. Then the R2 in 3-D with its LiDAR's beams at 20 cm above the floor.

## A3 · The track, and which sensor sees which obstacle

The track is MP11A's: a closed 21 m lap, 1.1 m wide, with a chicane, an S-bend and two long bends, 4 to 8 obstacles at least 1.6 m apart, drawn from a seed. Here the hard kinds are more likely (`mp.KINDS_B`: 15 % dark boxes, 30 % grey, 30 % low, 25 % black):

| kind | camera | depth camera | LiDAR |
|---|---|---|---|
| `box`, dark, 30 cm | sees it | sees it | sees it |
| `grey`, the floor's colour, 30 cm | **barely**: no dark pixels, only shading and its outline against the wall | sees it | sees it |
| `low`, dark, 10 cm | sees it, small | sees it beyond 0.6 m | **never**: it passes under the scan plane at 20 cm |
| `black`, glossy, 30 cm | sees it | **holes**: 85 % of its pixels return nothing | **dropouts**: 85 % of the beams on it return nothing |

No kind is invisible to all three, and no single sensor sees every kind well: their failures are **complementary**. That is the whole case for fusion.

In [ ]:
# One test track from above, and a pose in front of each kind of obstacle with what each sensor reports
tr = mp.track(1000, 3)
tr.describe()
fig, ax = plt.subplots(figsize=(10, 5.5)); rt.draw_track(tr, ax); plt.show()
for (so, do), ob in list(zip(tr.obstacle_s, tr.obstacles))[:4]:
    s = so - rt.WHEELBASE - 0.9
    x, y = tr.point(s, 0.0); th = tr.heading(s)
    t = rt.truth(tr, x, y, th)
    rng = np.random.default_rng(0)
    a = {"camera": rt.camera_rule(rt.render_rgb(tr, x, y, th, rng, 1.0)), "depth": rt.depth_rule(rt.render_depth(tr, x, y, th, rng)),
         "lidar": rt.lidar_rule(rt.render_scan(tr, x, y, th, rng))}
    print(f"{ob[3]:>5} {t['forward']:.2f} m ahead: " + ", ".join(f"{k} {'blocked' if v[0] else 'free'}" for k, v in a.items()))

**What you should see.** Test track 1000 of this project from above, at level 3 (light 0.53): four obstacles, a dark box, two grey boxes and a low block. Then, for each, what the three conventional rules (part C) say from 0.9 m before it: at the grey boxes the camera says free while the depth camera and the LiDAR say blocked; at the low block the LiDAR says free; only the dark box is seen by all three. Every sensor is wrong somewhere, and never all three at once.

## A4 · The noise, at five levels

MP11A's five seeded levels (all **ASSUMED**), here with what each does to the depth camera and the LiDAR too:

| level | the room | camera | depth camera | LiDAR |
|---|---|---|---|---|
| 0 | clean lab | 1 % pixel noise | 3 mm at 1 m | 0.5 % range noise |
| 1 | normal lab, 2 shadows, 1 sun patch | 2 %, light 0.85 - 1 | 1.5 × the noise, 2 % holes, half the sun's pixels lost | 1 %, 2 % dropouts |
| 2 | difficult day, 3 shadows, 2 sun patches | 3.5 %, light 0.70 - 0.85 | 2 ×, 5 % holes, 70 % of the sun's pixels lost | 1.5 %, 5 % dropouts |
| 3 | **dim room with sun**, 4 shadows, 3 sun patches | 5 %, light 0.50 - 0.65 | 3 ×, 10 % holes, 90 % of the sun's | 2 %, 10 % dropouts |
| 4 | extreme, 5 shadows, 4 sun patches | 7 %, light 0.35 - 0.50 | 4 ×, 15 % holes, all of the sun's | 3 %, 15 % dropouts |

A dim room hurts the camera only: the depth camera and the LiDAR bring their own light. Sun on the floor hurts the camera (bright patches) and the depth camera (its infrared pattern is drowned) but not the LiDAR. **You train at levels 0 to 3; the requirements hold at levels 2 and 3, on tracks you never train on; level 4 is never seen in training** and tells you how your fusion fails when the room changes.

In [ ]:
# The same place at levels 2, 3 and 4: the frame and the depth image
s0 = tr.obstacle_s[1][0] - 1.0
fig, ax = plt.subplots(2, 3, figsize=(13, 4.6))
for j, lv in enumerate((2, 3, 4)):
    t = mp.track(1000, lv)
    x, y = t.point(s0, 0.05); th = t.heading(s0)
    ax[0, j].imshow(rt.render_rgb(t, x, y, th, np.random.default_rng(lv), speed=1.5)); ax[0, j].axis("off")
    ax[0, j].set_title(f"level {lv}: light {t.light:.2f}", fontsize=9)
    d = rt.render_depth(t, x, y, th, np.random.default_rng(lv))
    ax[1, j].imshow(np.where(d > 0, d, np.nan), cmap="viridis_r", vmin=0, vmax=4); ax[1, j].axis("off")
    ax[1, j].set_title(f"depth: {100 * np.mean(d == 0):.0f} % of pixels without a reading", fontsize=9)
plt.tight_layout(); plt.show()

**What you should see.** Top, the frame darkening and blurring from level 2 to 4; bottom, the depth image of the same place, its holes growing (the share of pixels without a reading is printed above each).

## A5 · What the car must perceive, and how it drives

As in MP11A. **Blocked**: an obstacle's centre between the front axle and 1.5 m beyond it, within 0.55 m plus its radius to either side of the car's axis, and inside the colour camera's view (the zone all three sensors look into). **Where**: the nearest such obstacle's distance forward from the rear axle and sideways (left positive), in metres. Both come from the twin's geometry (`rt.truth`), exactly. Frames with an obstacle within 0.2 m of the zone's far end or within 5 degrees of the view's edge, where either answer is fair, are left out of the counts of missed obstacles and false stops.

The drive (`rt.drive`) decides ten times a second with the latest frame and depth image (30 a second) and the latest LiDAR scan (5.5 a second, so up to 0.18 s old); the potential field (`rt.FieldAround`) remembers every obstacle reported twice within a second and steers round it with a 10 cm margin; pure pursuit steers; the speed is held under 1 m/s² of sideways acceleration in the bends. A car that touches an obstacle stops the run. See MP11A, section A5, for every step.

## A6 · Why there is no exact answer, and what the ground truth is

No formula says which pixels, depth readings and LiDAR beams an obstacle produces in a room with shadows and sun, or how a lap unfolds when every decision depends on the readings before it. The ground truth is therefore numerical and exact to what it claims: **the labels** from the twin's geometry, and **the lap with perfect perception** - the same planner fed the exact labels - on every test track, converged in the time step of the integration (part C).

## A7 · What is asked

| quantity | where | required |
|---|---|---|
| **collisions** | the 10 test tracks of this project (seeds 1000 to 1009), never trained on, at noise levels 2 and 3, the potential field at 1.5 m/s: 20 laps | **none** |
| **lap time** | the same 20 laps | the mean within **5 %** of perfect perception's with the same planner and speed (part C) |
| **blocked or not** | 600 held-out poses at levels 2 and 3 (20 tracks never trained on) | for **every kind**, at most **5 %** missed; at most **5 %** false stops |
| **where** | the same poses, where both say blocked | the mean position error at most **10 cm** for every kind |
| **decision time** | one set of readings, on the Colab T4 | at most **10 ms**, a tenth of the 100 ms between decisions |

Reported, not required: noise level 4, never trained on; the conventional rules joined by OR and by their reliabilities, against your network; each sensor's contribution (what your network does with one sensor blanked).

**Why these numbers.** As in MP11A: zero collisions is the only acceptable safety result; 5 % on the lap time leaves room for a few false stops; an obstacle is seen in ten to fifteen readings on the way, so 5 % missed per reading is survivable; 10 cm is the planner's margin. The conventional rules joined by OR meet none of them at levels 2 and 3 (part C).

---

# Part B · The Data and the Physics

## B1 · Sensor fusion: a primer

The lectures did not cover sensor fusion. This section gives you what you need before you design a fusion network.

**Why fuse.** Each sensor measures a different physical quantity, so each fails for a different reason (the table in section A3). A camera sees light: it fails when an obstacle looks like the floor, when a shadow looks like an obstacle, and when the room is dark. A depth camera sees the distance to a projected infrared pattern: it fails where the pattern does not return (glossy or very dark surfaces), where the sun drowns it, and nearer than 0.6 m. A LiDAR sees the distance along one horizontal plane: it fails for anything below or above that plane and where its pulse does not return. When the failures are **complementary** - no situation defeats all of them at once - a fused perception can be better than every sensor alone. When they are not, fusion cannot invent what no sensor measured.

**Three ways to fuse.** They differ in *where* in the processing the sensors meet:

| | what is combined | example | for | against |
|---|---|---|---|---|
| **early fusion** | the raw data, into one input | the depth image as a fourth channel of the colour frame | one network sees everything at once, the cheapest network | the data must be aligned pixel by pixel; a LiDAR scan is not an image; one broken sensor corrupts the input |
| **late fusion** | each sensor's own decision, by a rule | three rules joined by OR, AND, a vote, or weighted by how far each can be trusted | simple, explainable; each sensor's pipeline is built and tested alone | each sensor decides alone: two weak hints (a faint outline and a few LiDAR returns) never add up to one strong one |
| **intermediate fusion** | features: each sensor has its own branch of layers, the branches' outputs are joined and a common head decides | a convolutional branch for the frame, one for the depth image, a 1-D convolutional branch for the scan, joined by a dense layer | each branch suits its data; the head learns which sensor to trust in which situation, and weak hints can add up | more weights; the network can learn to lean on one sensor and ignore the others |

Deep-learning systems mostly use **intermediate fusion**, and this project asks for it: **one network, three branches, heads for blocked and where.**

**The trap of learned fusion.** If one sensor is right most of the time in the training data, the network may learn to read that sensor alone - and fail when it fails. A common remedy is **modality dropout**: during training, now and then blank one sensor's input (a grey frame, an empty depth image, an empty scan), so that the network must learn to decide from the others too. A second remedy is to train in every condition the car will meet - here, all four training levels.

**A worked fusion rule.** Section B5 builds the simplest principled late fusion, by the rules' measured reliabilities, and part C scores it beside OR. Your network must beat both.

## B2 · What the network takes and returns

$$(\text{frame},\ \text{depth image},\ \text{scan})\ \longmapsto\ (\ell,\ f,\ s)$$

| input | as the R2 gives it | shape |
|---|---|---|
| frame | colour, uint8, 0 - 255 | `(n, 3, 64, 112)` |
| depth image | metres along the camera's axis, float32, **0 where there is no reading** | `(n, 48, 64)` |
| scan | metres, beam `i` at `i` degrees anticlockwise from straight ahead, float32, **0 where nothing returned** | `(n, 360)` |

and it returns, as in MP11A, the **logit** $\ell$ of *blocked* ($p = 1/(1 + e^{-\ell})$, blocked when $p > 0.5$) and **where** the nearest blocking obstacle stands, forward $f$ and sideways $s$ in metres from the rear axle.

The zeros mean *no reading*, not *distance zero*: a network that takes them as distances learns nonsense. Turning each into a value and a **mask** (1 where there is a reading) is the usual way, and it is your choice in part D.

**The loss** is MP11A's: the cross-entropy on blocked for every pose, and the Huber loss of the position (scaled by 0.5 m) on the blocked poses only.

**What can be built in.** The geometry of each sensor is known: a depth pixel and its distance give a point in the car's frame (`rt.depth_points`), a LiDAR beam and its range give a point on the floor plan. Feeding the network those positions, rather than leaving it to learn trigonometry, is a fair choice.

## B3 · The training data

`mp.training_data()` renders all three sensors from the twin: **100 tracks (seeds 0 to 99), noise levels 0 to 3, 40 poses on each: 16 000 poses**, with MP11A's labels (blocked, where, kind, edge, shadow) - about six minutes on one CPU thread.

In [ ]:
# The training data: frames, depth images and scans, rendered from the twin
t0 = time.perf_counter()
DATA = mp.training_data()
print(f"{len(DATA['blocked'])} poses in {time.perf_counter() - t0:.0f} s; blocked {100 * DATA['blocked'].mean():.1f} %; "
      + ", ".join(f"{k} {int(np.sum(DATA['kind'] == k))}" for k in rt.KIND_NAMES)
      + f"; depth pixels without a reading {100 * np.mean(DATA['depth'] == 0):.0f} %, beams without a return {100 * np.mean(DATA['scan'] == 0):.0f} %")
print(f"memory: frames {DATA['rgb'].nbytes / 1e6:.0f} MB, depth images {DATA['depth'].nbytes / 1e6:.0f} MB, scans {DATA['scan'].nbytes / 1e6:.0f} MB")

**What you should see.** 16 000 poses in 353 s on one CPU thread shared with other jobs; 48.6 % blocked: 1 065 dark boxes, 1 979 grey, 2 409 low and 2 316 black. 44 % of the depth pixels have no reading (beyond 8 m, nearer than 0.6 m, sun, glossy surfaces, noise) and 7 % of the LiDAR beams no return. The data take 564 MB of memory: 344 MB of frames, 197 MB of depth images, 23 MB of scans.

## B4 · The test data

* `mp.test_samples()`: **20 tracks (seeds 900 to 919), 15 poses each at noise levels 2, 3 and 4: 900 poses**, scored pose by pose; their labels are in `MP11B_truth.npz`, and the cell checks them;
* `mp.TEST_TRACKS`: **10 tracks (seeds 1000 to 1009)** of this project, driven in closed loop at levels 2, 3 and 4.

In [ ]:
# The held-out poses, and the check against the ground-truth file
G = np.load("MP11B_truth.npz")
TEST = mp.test_samples()
same = np.array_equal(TEST["blocked"], G["sample_blocked"]) and np.allclose(np.nan_to_num(TEST["where"]), np.nan_to_num(G["sample_where"]), atol=1e-5)
print(f"{len(TEST['blocked'])} held-out poses; labels as in the truth file: {same}")
for lv in mp.TEST_LEVELS:
    m = TEST["level"] == lv
    print(f"  level {lv}: {m.sum()} poses, blocked {int(TEST['blocked'][m].sum())}: "
          + ", ".join(f"{k} {int(np.sum(TEST['kind'][m] == k))}" for k in rt.KIND_NAMES) + f"; near the zone's edge {int(TEST['edge'][m].sum())}")

**What you should see.** 900 poses, labels as in the truth file: `True`. At each level 300 poses, 152 to 167 blocked, low blocks the most frequent kind; 56 to 69 poses per level have an obstacle near the zone's edge and are left out of the counts.

## B5 · A worked fusion rule: trust each sensor as far as it has earned

The simplest principled late fusion asks of each rule two numbers, measured on the training data: how often it says *blocked* when the path is blocked (its **hit** rate $h$), and how often it says *blocked* when the path is free (its **false** rate $f$). A rule that says *blocked* multiplies the odds of blocked by $h/f$; a rule that says *free* multiplies them by $(1 - h)/(1 - f)$:

$$\ln\frac{P(\text{blocked})}{P(\text{free})} = \ln\frac{P_0}{1 - P_0} + \sum_{\text{rules}} \begin{cases} \ln(h_i / f_i) & \text{rule } i \text{ says blocked} \\ \ln\big((1 - h_i)/(1 - f_i)\big) & \text{rule } i \text{ says free} \end{cases}$$

and the path is blocked when the odds exceed one. This is Bayes' rule with the sensors assumed independent (*naive Bayes*). A reliable rule ($h$ near 1, $f$ near 0) counts for much; an unreliable one for little. The position is taken from the most telling rule that says blocked. The cell measures the table and works one pose through by hand.

In [ ]:
# The worked fusion rule: the rules' reliabilities on the training data, and one pose worked through
sub = np.random.default_rng(0).choice(len(DATA["blocked"]), 4000, replace=False)      # 4000 training poses suffice
D4 = {k: v[sub] for k, v in DATA.items()}
ans_train = mp.rule_answers(D4)
TRUST = mp.trust_table(D4, ans_train)
for s_, t_ in TRUST.items():
    print(f"{s_:>6}: hit {100 * t_['hit']:5.1f} %, false {100 * t_['false']:5.1f} %  ->  says blocked: odds x {t_['hit'] / t_['false']:5.1f};"
          f" says free: odds x {(1 - t_['hit']) / (1 - t_['false']):.2f}")
i = int(np.where((D4["kind"] == "low") & (ans_train["lidar"][:, 0] == 0) & (ans_train["depth"][:, 0] == 1) & (ans_train["camera"][:, 0] == 0))[0][0])
lo = 0.0
print(f"\na pose with a low block {D4['where'][i][0]:.2f} m ahead: camera free, depth blocked, LiDAR free")
for s_ in mp.SENSORS:
    b = ans_train[s_][i, 0] > 0.5
    step = np.log(TRUST[s_]["hit"] / TRUST[s_]["false"]) if b else np.log((1 - TRUST[s_]["hit"]) / (1 - TRUST[s_]["false"]))
    lo += step
    print(f"  {s_:>6} says {'blocked' if b else 'free':>7}: log-odds {step:+.2f}, total {lo:+.2f}")
print(f"  probability of blocked {1 / (1 + np.exp(-lo)):.2f} (prior 0.5): {'blocked' if lo > 0 else 'free'}; OR says blocked, AND says free")

**What you should see.** The camera rule is the most telling when it speaks (hit 74.8 %, false 1.8 %: *blocked* multiplies the odds by 42.6), the depth rule next (55.7 % and 3.3 %, odds × 17.0), the LiDAR rule the least (64.0 % but 18.9 % false: it also reports obstacles that stand in the zone but out of the camera's view, which are labelled free; odds × 3.4). The worked pose, a low block 1.69 m ahead: the camera says free (log-odds -1.36), the depth camera blocked (+2.83), the LiDAR free (-0.81): total +0.66, probability 0.66, **blocked**. OR says blocked as well; AND would say free. Note what the rule cannot do: it treats the sensors as independent, so in a dim room, where the camera's reliability falls, it still trusts the camera as on an average day (part C measures what that costs).

---

# Part C · The Reference Model and the Ground Truth

## C1 · Two references, two jobs

| | the **reference model** | the **ground truth** |
|---|---|---|
| what it is | the conventional rules on the three sensors, joined by OR (Ex11's proposal) and by their reliabilities (section B5) | the twin's geometry: exact labels, and the lap driven with perfect perception |
| its job | to be **compared** with your network and **timed** per decision | to **score** both |
| planner | Ex11's steer-around rule at 1.0 m/s, and the potential field at 1.5 m/s | the potential field at 1.5 m/s, and the steer-around rule at 1.0 m/s |

## C2 · The three conventional rules, step by step

* **camera** (`rt.camera_rule`): Ex11's dark-pixel rule (MP11A, section C2): pixels darker than the frame's median floor by 0.15 in the zone; blocked when more than 0.4 % of the zone; the lowest dark pixel is the foot, placed on the flat floor.
* **depth** (`rt.depth_rule`): every depth pixel with a reading becomes a point in the car's frame (its distance along the camera's axis, its row and column, the camera's height and tilt); blocked when at least 6 points stand more than 4 cm above the floor inside the zone; the nearest of them, 8.5 cm further, is the obstacle's centre.
* **LiDAR** (`rt.lidar_rule`): every beam that returned becomes a point on the floor plan; blocked when any falls inside the zone; the nearest, 8.5 cm further, is the centre.

Each is a few array operations, as an engineer would write them; none was tuned against this project's test data.

In [ ]:
# The ground truth: the convergence of the perfect-perception lap, and the twin's physics checks
print("time step   mean lap over the 10 test tracks   smallest gap")
for dt, t, g in zip(G["conv_dt"], G["conv_time"], G["conv_gap"]):
    print(f"  {1e3 * dt:4.1f} ms     {t:.4f} s                          {100 * g:.2f} cm")
d = np.abs(np.diff(G["conv_time"]))
print("change between steps:", " ".join(f"{v:.1e}" for v in d), "s")
print(f"perfect perception, potential field at {float(G['speed'])} m/s: mean lap {G['field_time'].mean():.2f} s, "
      f"smallest gap {100 * G['field_gap'].min():.1f} cm, no collision: {bool(np.all(np.isfinite(G['field_time'])))}")
print(f"perfect perception, steer-around at {float(G['ref_speed'])} m/s: mean lap {G['steer_time'].mean():.2f} s")
checks = mp.check_sensors()

**What you should see.** The test tracks of this project place their obstacles exactly as MP11A's (only the kinds differ, and perfect perception does not depend on the kind), so the laps are MP11A's: 18.1345, 18.1254, 18.1243 and 18.1229 s at 4, 2, 1 and 0.5 ms, changes 9.1e-03, 1.0e-03 and 1.4e-03 s - no regular order, since the decisions every 0.1 s make the lap a non-smooth function of the step, but the 2 ms used is 2.4 ms (0.01 %) from the finest. Perfect perception: mean lap **18.13 s**, smallest gap 9.7 cm, no collision; steer-around at 1.0 m/s 28.54 s. The four physics checks as in MP11A: 0 m against Ex11's car, 2e-08 m (LiDAR) and 5e-08 m (depth) against the exact distances, an empty lap 0.6 % longer than L/v.

## C3 · The references on the held-out poses

The three rules, OR and the trust fusion of section B5 on the 900 held-out poses, level by level, and the time of one decision of the whole conventional pipeline (three rules and the fusion) on this runtime's CPU.

In [ ]:
# The conventional rules and their fusion on the held-out poses, level by level, and their time per decision
ans_test = mp.rule_answers(TEST)
ans_test["trust"], _ = mp.trust_fusion(ans_test, TRUST)
REF = {"rates": {}}
print(f"{'':>7}{'level':>6} " + " ".join(f"{'missed ' + k:>13}" for k in rt.KIND_NAMES) + f"{'false stops':>12}"
      + " ".join(f"{'where ' + k:>11}" for k in rt.KIND_NAMES))
for name in ("camera", "depth", "lidar", "OR", "trust"):
    for lv in mp.TEST_LEVELS:
        m = TEST["level"] == lv
        r = rt.rates(ans_test[name][m, 0].astype(int), {k: v[m] for k, v in TEST.items()}, ans_test[name][m, 1:])
        REF["rates"][(name, lv)] = r
        print(f"{name:>7}{lv:>6} " + " ".join(f"{100 * r['missed ' + k]:12.1f}%" for k in rt.KIND_NAMES)
              + f"{100 * r['false stops']:11.1f}%" + " ".join(f"{r['where ' + k + ' [cm]']:8.1f} cm" for k in rt.KIND_NAMES))
k0 = 0
one = {"rgb": TEST["rgb"][k0].transpose(1, 2, 0), "depth": TEST["depth"][k0], "scan": TEST["scan"][k0]}
REF["t_rule_s"] = middle(lambda: mp.trust_perceive(TRUST)(None, one))
print(f"one decision of the conventional pipeline (three rules and their fusion): {REF['t_rule_s'] * 1e3:.3f} ms")

**What you should see.** Each rule fails where section A3 says: the camera misses 84 to 89 % of the grey boxes at levels 2 and 3 and breaks down at level 4 (75 % false stops); the depth rule misses 15 to 64 % of every kind (holes, the 0.6 m minimum, a 4 cm height threshold over a noisy floor) but places what it finds within 2 to 4 cm; the LiDAR misses every low block (100 %), 12 to 28 % of the black ones, and stops on 16 to 27 % of free poses (obstacles in the zone but outside the camera's view). **OR** misses nothing at levels 2 and 3 but stops on 16.5 and 27.0 % of the free poses, and its positions at level 3 are 21 to 25 cm off for box, low and black (it takes the nearest report, often the camera's in the dim room). The **trust fusion** cuts the false stops to 3.5 - 4.6 % but misses 39 to 52 % of the grey boxes: the camera's strong *free* outweighs the others. One decision of the conventional pipeline takes 0.273 ms on this runtime's CPU.

## C4 · The references in closed loop

Each of the 10 test tracks once at levels 2 and 3: the rules joined by OR with the steer-around rule at 1.0 m/s and with the potential field at 1.5 m/s, and the trust fusion with the field at 1.5 m/s. About four minutes.

In [ ]:
# The conventional pipelines driving the test tracks, against perfect perception
t0 = time.perf_counter()
print("rules joined by OR, steer-around at 1.0 m/s:")
REF["or_steer"] = mp.closed_loop(lambda tr: rt.conventional(("camera", "depth", "lidar")), rt.SteerAround, mp.REF_SPEED, verbose=True)
print("rules joined by OR, potential field at 1.5 m/s:")
REF["or_field"] = mp.closed_loop(lambda tr: rt.conventional(("camera", "depth", "lidar")), rt.FieldAround, mp.SPEED, verbose=True)
print("rules joined by their reliabilities, potential field at 1.5 m/s:")
REF["trust_field"] = mp.closed_loop(lambda tr: mp.trust_perceive(TRUST), rt.FieldAround, mp.SPEED, verbose=True)
REF["perfect_time"] = float(G["field_time"].mean())
print(f"perfect perception, potential field at 1.5 m/s: mean lap {REF['perfect_time']:.2f} s ({time.perf_counter() - t0:.0f} s for the 60 laps)")

**What you should see.** OR with the steer-around rule at 1.0 m/s collides on 5 and 8 of the 10 tracks at levels 2 and 3; OR with the field at 1.5 m/s on 0 and 5 (its false stops cost time - 18.45 and 18.69 s - and at level 3 its wrong positions send the car into obstacles); the trust fusion with the field on 0 and 5 (17.86 and 18.57 s). Perfect perception: 18.13 s, no collision. The 60 laps took 307 s here.

## C5 · A lap with perfect perception, filmed

The R2 in 3-D with its LiDAR's beams, its colour frame and its depth image, and the track from above: a few seconds of test track 1000 at noise level 3, perfect perception and the potential field at 1.5 m/s. About a minute to draw.

In [ ]:
# The film: a few seconds of a lap with perfect perception, in the dim room
from IPython.display import HTML, display
tr = mp.track(1000, 3)
run = rt.drive(tr, rt.oracle(tr), rt.FieldAround(), mp.SPEED, need=(), record=True)
anim = rt.film(run, tr, title="perfect perception, potential field, 1.5 m/s, level 3", start=1.0, seconds=5.0, every=5)
display(HTML(anim.to_html5_video()))                    # needs ffmpeg, which Colab has

**What you should see.** The R2 in the dim room, its LiDAR beams sweeping past the grey and black obstacles, the depth image with its holes, and its path on the track from above.

---

# Part D · The Network

## D1 · What is given, and what is yours

**Yours** are three things, each a TODO cell: the **fusion network** (TODO 1), its **training** (TODO 2), and the **figures** you need to understand it (TODO 3). Everything else is given.

**The contract.** Your network is a PyTorch module called `net`, used as `net(frame, depth, scan)`:

| argument | meaning | shape |
|---|---|---|
| `frame` | colour frames, uint8, 0 - 255, on `DEVICE` | `(n, 3, 64, 112)` |
| `depth` | depth images, metres, float32, 0 = no reading | `(n, 48, 64)` |
| `scan` | LiDAR scans, metres, float32, 0 = no return, beam `i` at `i` degrees anticlockwise from ahead | `(n, 360)` |
| returns | the logit of *blocked*, then *forward* and *sideways* in metres | `(n, 3)` |

Every scaling, mask and change of coordinates happens inside `net`: the drive and the evaluation hand it the readings as the sensors give them.

## D2 · The loss and a quick score, given

`loss_fn(out, blocked, where)` is MP11A's: the cross-entropy on every pose and the masked Huber loss of the position, scaled by 0.5 m. `predict(net, D)` runs `net` on a dict of readings in batches; `quick_score()` scores `net` on the held-out poses of levels 2 and 3 in a few seconds.

In [ ]:
# The loss and the quick score, given ------------------------------------------------
import torch.nn.functional as F_

def loss_fn(out, blocked, where):
    """Cross-entropy on blocked (every pose) + Huber loss of the position (blocked poses only), scaled by 0.5 m."""
    l_b = F_.binary_cross_entropy_with_logits(out[:, 0], blocked)                       # L_blocked: is the path blocked?
    m = blocked > 0.5                                                                  # the position exists only where blocked
    l_w = F_.smooth_l1_loss(out[m, 1:] / 0.5, where[m] / 0.5, beta=0.1) if m.any() else out.sum() * 0   # L_where, Huber
    return l_b + l_w, l_b.detach(), l_w.detach()

def predict(net, D, batch=256):
    """The network's answers on a dict of readings: probability blocked, forward, sideways (NumPy)."""
    out = []
    with torch.no_grad():
        for i in range(0, len(D["blocked"]), batch):
            o = net(torch.as_tensor(D["rgb"][i:i + batch], device=DEVICE), torch.as_tensor(D["depth"][i:i + batch], device=DEVICE),
                    torch.as_tensor(D["scan"][i:i + batch], device=DEVICE))                # one batch through the network
            out.append(o.float().cpu().numpy())
    o = np.concatenate(out)
    return 1 / (1 + np.exp(-o[:, 0])), o[:, 1:]

def quick_score(threshold=mp.THRESHOLD):
    """Missed per kind, false stops and position error per kind [cm] on the held-out poses of levels 2 and 3."""
    m = np.isin(TEST["level"], mp.REQUIRED_LEVELS)
    Dm = {k: v[m] for k, v in TEST.items()}
    p, w = predict(net, Dm)
    return rt.rates((p > threshold).astype(int), Dm, w)

## D3 · TODO 1 · Your fusion network

Write the class `Net` and make `net = Net().to(DEVICE)`: **one network, three branches, heads for blocked and where** (intermediate fusion, section B1). Decide:

* **each branch**: MP11A's convolutional network suits the frame; the depth image is a small image (a 2-D convolutional branch); the scan is a ring of 360 values (a 1-D convolution, perhaps with circular padding, since beam 359 is beside beam 0);
* **what each branch sees**: the zeros of the depth image and the scan as a mask channel; the scan turned so that straight ahead is in the middle; perhaps each beam's position on the floor plan (section B2);
* **the fusion**: the branches' features joined (concatenated), a dense layer or two, then the three outputs.

In [ ]:
# TODO 1 - your fusion network. net(frame, depth, scan) -> (n, 3): logit, forward [m], sideways [m]
import torch.nn as nn

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        ...                                              # your three branches and the fusion

    def forward(self, frame, depth, scan):
        ...                                              # return (n, 3)

torch.manual_seed(88)
net = Net().to(DEVICE)
print("weights:", sum(p.numel() for p in net.parameters()))

## D4 · TODO 2 · Your training

Train `net` on `DATA` (section B3) with `loss_fn`. Decide the optimiser, its schedule, the batch size, the number of passes, and **whether and how to blank a sensor at random** (modality dropout, section B1). Keep the history of the two loss terms for TODO 3 and **measure the training time**: set `TRAIN_SECONDS`. `quick_score()` tells you where you stand.

In [ ]:
# TODO 2 - your training
history = {"blocked": [], "where": []}
t0 = time.perf_counter()
...                                                      # your optimiser and loop; append each term to history
TRAIN_SECONDS = time.perf_counter() - t0
print(f"trained in {TRAIN_SECONDS:.0f} s;", {k: round(v, 3) for k, v in quick_score().items()})

## D5 · TODO 3 · Your figures

Draw what you need to understand your network and to answer the report: at least the two loss terms during training, and **what each sensor contributes** - the share missed per kind with one sensor blanked at a time (a grey frame, an empty depth image, an empty scan). Save each figure you want in the report as `MP11B_report_<name>.png` (in `cc.output_path`).

In [ ]:
# TODO 3 - your figures
...

## D6 · The evaluation, given

This cell asks your trained `net` the questions part E scores, and keeps the answers in `R`:

* its answers on the 900 held-out poses (levels 2, 3 and 4);
* **one lap of each of the 10 test tracks at levels 2, 3 and 4**, the potential field at 1.5 m/s: lap times, smallest gaps, collisions (about two minutes);
* the **decision time** on this runtime: one set of readings through the network on `DEVICE`, against the conventional pipeline's (part C).

Set `TRAIN_SECONDS` to the training time you measured.

In [ ]:
# The evaluation, given ------------------------------------------------------------
net.eval()
p_test, w_test = predict(net, TEST)                              # the held-out poses, levels 2 - 4
t0 = time.perf_counter()
loop = mp.closed_loop(lambda tr: mp.network_perceive(net, DEVICE), rt.FieldAround, mp.SPEED, levels=mp.TEST_LEVELS, verbose=True)
print(f"({time.perf_counter() - t0:.0f} s for the {loop['time'].size} laps)")
one = [torch.as_tensor(TEST[k][:1], device=DEVICE) for k in ("rgb", "depth", "scan")]
with torch.no_grad():
    t_net = middle(lambda: net(*one))                            # one decision of the network
print(f"one decision: network {t_net * 1e3:.3f} ms on {DEVICE}, conventional pipeline {REF['t_rule_s'] * 1e3:.3f} ms on the CPU "
      f"(the network takes {t_net / REF['t_rule_s']:.1f} times as long)")
R = {"p_test": p_test, "w_test": w_test, "loop": loop, "t_net_s": t_net, "train_seconds": TRAIN_SECONDS,
     "n_weights": sum(p.numel() for p in net.parameters()), "device": DEVICE}   # part E reads the answers from here
torch.save(net.state_dict(), cc.output_path("MP11B_net.pt"))     # the trained weights, beside the report

---

# Part E · The Report

## E1 · What this part does

It scores your network's answers, kept by section D6 in `R`, against the ground truth (the labels `TEST` and the perfect-perception laps in `G`) and the references `REF` from part C: the goal metrics (TODO 4) and the report's figures (TODO 5). Then come the eighteen questions.

In [ ]:
# What part E uses, all from the parts above
print("in R:", ", ".join(R))
print(f"perfect perception: mean lap {REF['perfect_time']:.2f} s; conventional pipeline {REF['t_rule_s'] * 1e3:.3f} ms per decision")

## E2 · TODO 4 · The goal metrics

Compute and put in `METRICS` (a name and a number each):

* on the held-out poses of levels 2 and 3, pooled: the share missed of each kind, the false stops and the mean position error of each kind - for your network, the rules joined by OR and the trust fusion;
* the same for your network at level 4, never trained on;
* in closed loop: the collisions at levels 2 and 3 (of 20) and at level 4 (of 10); the mean lap at levels 2 and 3 against perfect perception's, in per cent; the collisions of the two references;
* the decision time of the network, the conventional pipeline's, their ratio; the offline training time.

Then say in one line each whether the five requirements of part A are met.

In [ ]:
# TODO 4 - the goal metrics
METRICS = {}
...
for k, v in METRICS.items():
    print(f"{k:>60}: {v:.2f}")

## E3 · TODO 5 · The report's figures

Draw, and save as `MP11B_report_<name>.png`, the figures your answers refer to. At least: the share missed per kind and level for your network, OR and the trust fusion; the lap times per track at each level against perfect perception's; and the path of one lap at level 3 on the track from above.

In [ ]:
# TODO 5 - the report's figures
...

## E4 · The report

Answer the eighteen questions below, each in a few sentences **with your numbers**; an answer needs at least fifteen words. Questions 1 to 4 are about your network and its training, 5 to 9 about what it perceives, 10 to 14 about the car it drives, 15 to 17 ask you to use it, 18 concludes. The next cell writes `MP11B_report.md` with the results table above and your answers, and the cell after it turns the report into a PDF. **Upload the PDF.**

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "1. Your fusion network: each branch (its input, layers and features), how the branches are joined, the head, and the number of weights. Is it early, late or intermediate fusion, and why did you choose it?": """
""",
    "2. How does your network treat the depth camera's holes and the LiDAR's missing returns (the zeros)? What would go wrong if it took them as distances of zero? Give a number if you tried.": """
""",
    "3. Your training: optimiser, schedule, batch, passes, the modality dropout (if any), and the offline time on the T4. Train once without the dropout: how do the misses per kind change?": """
""",
    "4. The threshold on blocked: which did you use, and how do the misses and false stops at levels 2 and 3 change at 0.3 and 0.7?": """
""",
    "5. The share missed of each kind at levels 2 and 3, for your network, OR and the trust fusion. Which kind is hardest for each, and why?": """
""",
    "6. What each sensor contributes: your misses per kind with the frame, the depth image and the scan blanked in turn. Which sensor does your network lean on for grey boxes, for low blocks and for black ones? Does that match the table of section A3?": """
""",
    "7. False stops at levels 2 and 3 for your network, OR and the trust fusion. Why does OR stop so often, and what does the trust fusion do about it?": """
""",
    "8. The mean position error of each kind for your network and for OR. Which sensor gives the best position, and does your network reach it?": """
""",
    "9. Level 4, never trained on: your misses, false stops and position errors there. What changed in each sensor's readings, and which of your network's answers broke first?": """
""",
    "10. Closed loop at levels 2 and 3: collisions (of 20), the mean lap, and the per cent against perfect perception's. Which tracks were slowest, and why?": """
""",
    "11. The references in closed loop: OR with the steer-around rule at 1.0 m/s, OR with the field at 1.5 m/s, the trust fusion with the field. Their collisions, and on which kind of obstacle?": """
""",
    "12. Closed loop at level 4: how many collisions, and what caused them - missed obstacles, phantom obstacles or wrong positions? Show one lap's path.": """
""",
    "13. The worked fusion rule of section B5 multiplies the odds by each rule's hit / false ratio. Which assumption of it is wrong for these sensors (think of the dim room, and of a grey box), and how does your network avoid it?": """
""",
    "14. Remove one sensor for good: retrain (or evaluate) your network with the depth camera blanked, as if the R2 had only a camera and a LiDAR. Collisions and misses per kind at levels 2 and 3?": """
""",
    "15. Faster: drive the test tracks at levels 2 and 3 at 1.8 m/s. Collisions, mean lap and the per cent against perfect perception's at 1.8 m/s. Is it worth it?": """
""",
    "16. The decision time of your network on the T4 and on the CPU, the conventional pipeline's, and their ratio. Where does your network spend its time - which branch is the largest?": """
""",
    "17. Train again with another seed. How much do the misses per kind, the position errors and the closed-loop results change? Is your result reproducible within the requirements?": """
""",
    "18. To conclude: which sensor would you keep if the R2 could carry only two, given your measurements in the dim room with sun? And what does a fusion network do that the rules joined by OR cannot?": """
""",
}

In [ ]:
MIN_WORDS = 15                                              # an answer is a few sentences with numbers, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:

    rows = ["| metric | value |", "|---|---|"] + [f"| {k} | {v:.2f} |" for k, v in METRICS.items()]
    out = ["# MP11B — Fusion of Camera, Depth Camera and LiDAR", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("MP11B_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figures saved as `MP11B_report*.png`, and downloads it.

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns MP11B_report.md into MP11B_report.pdf, with any figure
# saved as MP11B_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "MP11B_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "MP11B_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("MP11B_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "MP11B_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass